# Comprehensions, classes, generators, and decorators: practice

Filter books → build structured classes → yield unique authors → understand inheritance/hints → recognize decorator registration.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [guides/10_intermediate_python.ipynb](../../../../guides/10_intermediate_python.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Variables, dictionaries, and functions

A function consumes an input and returns a value. Dictionary lookup with get can handle optional fields. Comprehensions transform a collection.

**Predict:** Which book has no author?

In [ ]:
books = [{"title": "Python", "author": "Ada"}, {"title": "Agents"}]
def has_author(book):
    return bool(book.get("author"))
titles = [book["title"] for book in books]
missing = [book["title"] for book in books if not has_author(book)]
print(titles, missing)
assert missing == ["Agents"]

**Check:** Agents. get returns None for the missing optional key; bracket lookup requires the key to exist.

**Change one thing:** Add another authored book. Collect unique authors with a set and explain how it differs from a list.

## 2. Classes, methods, and generators

A class bundles data and behavior. A method receives self, the current instance. A generator yields values as they are requested.

**Predict:** Will the same author be yielded twice?

In [ ]:
from dataclasses import dataclass
@dataclass
class Book:
    title: str
    author: str | None = None
    def has_author(self):
        return bool(self.author)
class BookShelf:
    def __init__(self, books):
        self.books = books
    def unique_authors(self):
        seen = set()
        for book in self.books:
            if book.has_author() and book.author not in seen:
                seen.add(book.author)
                yield book.author
shelf = BookShelf([Book("Python", "Ada"), Book("Agents", "Ada"), Book("Notes")])
print(list(shelf.unique_authors()))
assert list(shelf.unique_authors()) == ["Ada"]

**Check:** No. The seen set prevents duplicates.

**Change one thing:** Add a book by another author. Explain why yield allows iteration to continue while return would end it.

## 3. Streaming sends partial progress

A generator yields multiple updates. A streamed API can include multiple event types, so a UI should select the text events rather than assume every event is text.

**Predict:** How many text chunks are selected?

In [ ]:
events = [
    {"type": "status", "value": "started"},
    {"type": "text", "value": "Hello "},
    {"type": "text", "value": "learner"},
]
def text_chunks(items):
    for event in items:
        if event["type"] == "text":
            yield event["value"]
chunks = list(text_chunks(events))
print(chunks, "→", "".join(chunks))
assert "".join(chunks) == "Hello learner"

**Check:** Two. yield preserves progress; return ends an ordinary function.

**Change one thing:** Insert a non-text event between the chunks. Confirm it does not become displayed answer text.

## 4. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 5. Generated code is an output before it is execution

A coding workflow designs, writes, and reviews code. Generated files or Docker execution are side effects. First inspect and validate a harmless code string locally.

**Predict:** Does parsing Python text execute it?

In [ ]:
import ast
proposed = "def add(a, b):\n    return a + b\n"
tree = ast.parse(proposed)  # Parse syntax only; do not execute generated text.
functions = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
print(functions)
assert functions == ["add"]
checks = {"has_function": True, "has_test_plan": False}
print("Review:", checks)

**Check:** No. Syntax validity does not prove behavior or make generated code safe to execute.

**Change one thing:** Write a test plan for the intended function. Explain which separate step would run code and where the source uses Docker.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.